# Activity Prediction Notebook

This notebook keeps only the code needed to train the Section 1.3 activity recognizer, validate it on the training set, and generate predictions for the test set.

In [6]:
from pathlib import Path

import joblib
import pandas as pd

from mhealth_activity import Recording
from mhealth_activity.activity_recognition import activity_f1_summary, build_activity_recognizer, evaluate_activity_recognizer

TRAIN_DIR = Path(r"c:\Users\giaco\Desktop\ETH\Mobile_Health\DATA\data\train")
TEST_DIR = Path(r"c:\Users\giaco\Desktop\ETH\Mobile_Health\DATA\data\test")
ACTIVITY_ORDER = ["standing", "walking", "running", "cycling"]

TRAIN_DIR, TRAIN_DIR.exists(), TEST_DIR.exists()

(WindowsPath('c:/Users/giaco/Desktop/ETH/Mobile_Health/DATA/data/train'),
 True,
 True)

## Train Activity Recognizer

In [12]:
activity_recognizer = build_activity_recognizer(TRAIN_DIR)
#activity_recognizer

## Save Trained Recognizer

Save the trained activity recognizer once so it can be loaded later without retraining.

In [8]:
activity_model_path = Path("activity_recognizer.joblib")
joblib.dump(activity_recognizer, activity_model_path)
activity_model_path.resolve()

WindowsPath('C:/Users/giaco/Desktop/ETH/Mobile_Health/III_Project/MHex3/activity_recognizer.joblib')

## Optional Training-Set Check

In [9]:
"""activity_eval_df = evaluate_activity_recognizer(activity_recognizer, TRAIN_DIR)
activity_metric_series = activity_f1_summary(activity_eval_df)
activity_metric_series.rename("f1").to_frame()
"""

'activity_eval_df = evaluate_activity_recognizer(activity_recognizer, TRAIN_DIR)\nactivity_metric_series = activity_f1_summary(activity_eval_df)\nactivity_metric_series.rename("f1").to_frame()\n'

## Predict Test Set

In [10]:
def parse_trace_id(path: Path) -> int:
    return int(path.stem.split("_")[-1])


test_predictions = []
for path in sorted(TEST_DIR.glob("*.pkl")):
    recording = Recording(str(path))
    predicted_activities = activity_recognizer.predict_activities(recording)
    test_predictions.append(
        {
            "Id": parse_trace_id(path),
            **{activity: bool(predicted_activities[activity]) for activity in ACTIVITY_ORDER},
        }
    )

activity_submission_df = pd.DataFrame(test_predictions).sort_values("Id").reset_index(drop=True)
activity_submission_df.head()

,Id,standing,walking,running,cycling
0,0,True,True,True,False
1,1,False,True,False,True
2,2,True,True,False,True
3,3,False,True,False,False
4,4,False,True,False,False


In [ ]:

activity_submission_path = Path("activity_predictions.csv")
activity_submission_df.to_csv(activity_submission_path, index=False)
activity_submission_path.resolve(), activity_submission_df.shape


(WindowsPath('C:/Users/giaco/Desktop/ETH/Mobile_Health/III_Project/MHex3/activity_predictions.csv'),
 (280, 5))